In [1]:
%load_ext autoreload
%autoreload 2

import cv2
import numpy as np
import json
import math
from pathlib import Path
from PIL import Image

# Загружаем персонажа и разметку
rgba = cv2.imread("OUTPUT/img01_clean.png", cv2.IMREAD_UNCHANGED)
with open("OUTPUT/img01_clean_pose.json") as f:
    pose = json.load(f)

skeleton = pose["skeleton"]
keypoints = {name: tuple(pt) for name, pt in pose["keypoints"].items()}

print(f"Изображение: {rgba.shape[1]}x{rgba.shape[0]}")
print(f"Точек: {len(keypoints)}, костей: {len(skeleton['bones'])}")

Изображение: 396x544
Точек: 8, костей: 7


In [2]:
h, w = rgba.shape[:2]
mask = rgba[:, :, 3] > 0

# Координаты всех пикселей
yy, xx = np.mgrid[:h, :w]
pixel_coords = np.stack([xx, yy], axis=-1).astype(float)

def point_to_segment_dist(p, a, b):
    """Расстояние от каждого пикселя до отрезка (кости)"""
    ab = b - a
    ap = p - a
    t = np.sum(ap * ab, axis=-1) / (np.sum(ab * ab) + 1e-8)
    t = np.clip(t, 0, 1)
    proj = a + t[..., None] * ab
    return np.sqrt(np.sum((p - proj) ** 2, axis=-1))

# Для каждой кости считаем расстояние от каждого пикселя
bone_dists = []
bone_names = []
for a_idx, b_idx in skeleton['bones']:
    a = np.array(keypoints[skeleton['points'][a_idx]], dtype=float)
    b = np.array(keypoints[skeleton['points'][b_idx]], dtype=float)
    bone_dists.append(point_to_segment_dist(pixel_coords, a, b))
    bone_names.append(f"{skeleton['points'][a_idx]}-{skeleton['points'][b_idx]}")

bone_dists = np.stack(bone_dists, axis=-1)
bone_labels = np.argmin(bone_dists, axis=-1)  # каждый пиксель → ближайшая кость

# Вырезаем части
parts = {}
colors = [(255,100,100), (100,255,100), (100,100,255),
          (255,255,100), (255,100,255), (100,255,255), (200,200,100)]

vis = np.zeros((h, w, 3), dtype=np.uint8)
for i, name in enumerate(bone_names):
    part_mask = (bone_labels == i) & mask
    part_rgba = rgba.copy()
    part_rgba[~part_mask] = 0
    parts[name] = {'image': part_rgba, 'mask': part_mask, 'bone_idx': i}
    vis[part_mask] = colors[i % len(colors)]

# Показываем сегменты
cv2.imwrite("OUTPUT/img03_segments.png", vis)
print(f"Сегментов: {len(parts)}")
for name, p in parts.items():
    print(f"  {name:20s}  пикселей: {np.sum(p['mask'])}")


Сегментов: 7
  head-neck             пикселей: 1777
  neck-left_hand        пикселей: 492
  neck-right_hand       пикселей: 533
  neck-hip              пикселей: 437
  hip-left_foot         пикселей: 303
  hip-right_foot        пикселей: 520
  hip-tail              пикселей: 23


In [3]:
def rotate_part(part_img, pivot, angle_deg):
    h, w = part_img.shape[:2]
    M = cv2.getRotationMatrix2D(pivot, angle_deg, 1.0)
    return cv2.warpAffine(part_img, M, (w, h),
                          flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_CONSTANT,
                          borderValue=(0, 0, 0, 0))

def render_frame(parts, keypoints, skeleton, bone_names, angles):
    h, w = list(parts.values())[0]['image'].shape[:2]
    # Белый фон + полная непрозрачность
    frame = np.full((h, w, 4), 255, dtype=np.uint8)

    for i, name in enumerate(bone_names):
        a_idx, b_idx = skeleton['bones'][i]
        pivot = keypoints[skeleton['points'][a_idx]]
        angle = angles.get(name, 0)
        rotated = rotate_part(parts[name]['image'], pivot, angle)
        fg_mask = rotated[:, :, 3] > 0
        frame[fg_mask] = rotated[fg_mask]

    return frame

# Генерируем кадры — Пикачу машет лапами и хвостом
num_frames = 30
frames = []

for f in range(num_frames):
    t = f / num_frames
    wave = math.sin(t * 2 * math.pi)

    angles = {}
    for name in bone_names:
        angles[name] = 0

    # Лапы машут
    angles["neck-left_hand"] = wave * 15
    angles["neck-right_hand"] = -wave * 15
    # Ноги покачиваются
    angles["hip-left_foot"] = wave * 8
    angles["hip-right_foot"] = -wave * 8
    # Хвост виляет
    angles["hip-tail"] = wave * 12

    frame = render_frame(parts, keypoints, skeleton, bone_names, angles)
    frames.append(frame)

# Сохраняем GIF
pil_frames = []
for frame in frames:
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGRA2RGB)
    pil_frames.append(Image.fromarray(rgb))

gif_path = "OUTPUT/img01_animated.gif"
pil_frames[0].save(
    gif_path,
    save_all=True,
    append_images=pil_frames[1:],
    duration=50,  # мс на кадр (20 fps)
    loop=0
)

print(f"GIF сохранён: {gif_path}")
print(f"Кадров: {len(frames)}, размер: {frames[0].shape[1]}x{frames[0].shape[0]}")


GIF сохранён: OUTPUT/img01_animated.gif
Кадров: 30, размер: 396x544
